Khushi Khatri beb222 exp3

Aim:apply various other text preprocessing techniques for any given text:stop word removal,lemmatization/stemming
Text Preprocessing: Stop Word Removal, Stemming, and Lemmatization

Theory

Raw text data collected from real-world sources (such as user reviews, social media posts, or documents) is unstructured and contains a lot of noise — filler words, grammatical variations, and inconsistent word forms — that add little value to text analysis or Natural Language Processing (NLP) tasks. Text preprocessing is the essential first step in any NLP pipeline that transforms raw text into a clean, standardized format suitable for machine learning models, information retrieval, or text mining. This experiment focuses on two key preprocessing techniques: Stop Word Removal and Stemming/Lemmatization.

1. Stop Word Removal

Stop words are common words in a language (such as "the," "is," "at," "which," "on," "and," "a") that occur very frequently in text but carry little to no meaningful semantic information for tasks like text classification, sentiment analysis, or search indexing. Since these words appear in nearly every sentence regardless of context, retaining them increases the dimensionality of the data without adding discriminative value, and can slow down downstream algorithms.

Stop word removal is the process of filtering out these high-frequency, low-information words from a tokenized text, leaving behind only the content-bearing words. For example:

Original: "The place was very cozy and the check-in was smooth"
After stop word removal: "place cozy check-in smooth"

Libraries like NLTK provide predefined stop word lists for various languages (e.g., nltk.corpus.stopwords). While effective, stop word removal must be applied carefully, since in some contexts (e.g., sentiment analysis, where words like "not" are critical) removing certain stop words can distort meaning.

2. Stemming

Stemming is a rule-based, crude heuristic process that reduces a word to its root or base form (stem) by chopping off prefixes or suffixes, without necessarily producing a valid dictionary word. It is fast and computationally inexpensive, making it suitable for large-scale text processing where speed matters more than linguistic precision.

The most widely used stemming algorithm is the Porter Stemmer, which applies a series of suffix-stripping rules. For example:

"amazing" → "amaz"
"studies" → "studi"
"connection," "connected," "connecting" → "connect"

Because stemming operates purely on word surface patterns, it can sometimes produce non-words or over-stem/under-stem (e.g., merging unrelated words or failing to reduce related words to the same stem), reducing linguistic accuracy.

3. Lemmatization

Lemmatization is a more linguistically informed process that reduces a word to its lemma — its proper dictionary/base form — by using vocabulary and morphological analysis (often with the help of Part-of-Speech, or POS, tagging) rather than simple suffix stripping. Unlike stemming, lemmatization always produces valid words.

For example:

"amazing" (adjective) → "amazing"
"studies" (verb) → "study"
"better" (adjective, with POS context) → "good"

Lemmatization typically uses lexical databases such as WordNet, and its accuracy improves significantly when combined with POS tagging, since the correct lemma of a word often depends on its grammatical role in the sentence (e.g., "saw" as a noun vs. a verb).

In [1]:
import re
import string
import pandas as pd
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
print('Setup complete.')

[nltk_data] Downloading package punkt to /home/computer/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/computer/nltk_data...
[nltk_data] Downloading package omw-1.4 to /home/computer/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


Setup complete.


In [2]:
df = pd.read_csv('Balanced_Airbnb_Reviews_Dataset.csv')
print('Shape:', df.shape)
df[['review_id', 'review_text']].head()

Shape: (15000, 42)


,review_id,review_text
0,369314882,Amazing stay! The place felt very cozy for 4 g...
1,490116563,It was okay for the price. Location in XIII Au...
2,582235668,Loved every minute of it. Our superhost was su...
3,68054683,Decent stay overall. Some things could be impr...
4,248483824,Reasonable for a short trip. Location in Long ...


In [3]:
def tokenize_text(text):
    """Return sentence tokens and word tokens for a given piece of text."""
    if not isinstance(text, str) or text.strip() == '':
        return [], []
    sentences = sent_tokenize(text)
    words = word_tokenize(text)
    return sentences, words

def remove_stopwords(tokens):
    """Remove English stopwords from a list of tokens."""
    return [tok for tok in tokens if tok.lower() not in stop_words]

def filter_tokens(tokens):
    """Lowercase, remove punctuation, numbers, stopwords, and short tokens."""
    cleaned = []
    for tok in tokens:
        tok = tok.lower()
        if tok in string.punctuation:
            continue
        tok = tok.translate(str.maketrans('', '', string.punctuation))
        if tok == '':
            continue
        if tok.isdigit():
            continue
        if tok in stop_words:
            continue
        if len(tok) < 2:
            continue
        cleaned.append(tok)
    return cleaned

In [4]:
sample_text = df['review_text'].iloc[0]
sample_tokens = word_tokenize(sample_text)

print('Original Tokens:\n', sample_tokens)
print('\nAfter Stopword Removal:\n', remove_stopwords(sample_tokens))

Original Tokens:
 ['Amazing', 'stay', '!', 'The', 'place', 'felt', 'very', 'cozy', 'for', '4', 'guests', '.', 'Check-in', 'was', 'smooth', 'and', 'the', 'amenities', 'were', 'exactly', 'what', 'we', 'needed', '.']

After Stopword Removal:
 ['Amazing', 'stay', '!', 'place', 'felt', 'cozy', '4', 'guests', '.', 'Check-in', 'smooth', 'amenities', 'exactly', 'needed', '.']


In [6]:
def get_wordnet_pos(tag):
    """Map NLTK POS tag to format WordNetLemmatizer accepts."""
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [7]:
def stem_tokens(tokens):
    """Apply Porter Stemming to a list of tokens."""
    return [stemmer.stem(tok) for tok in tokens]

sample_filtered = filter_tokens(sample_tokens)
print('Filtered Tokens:\n', sample_filtered)
print('\nStemmed Tokens:\n', stem_tokens(sample_filtered))

Filtered Tokens:
 ['amazing', 'stay', 'place', 'felt', 'cozy', 'guests', 'checkin', 'smooth', 'amenities', 'exactly', 'needed']

Stemmed Tokens:
 ['amaz', 'stay', 'place', 'felt', 'cozi', 'guest', 'checkin', 'smooth', 'amen', 'exactli', 'need']


In [8]:
def lemmatize_tokens(tokens):
    """Apply WordNet Lemmatization with POS tagging to a list of tokens."""
    pos_tags = nltk.pos_tag(tokens)
    return [lemmatizer.lemmatize(tok, get_wordnet_pos(pos)) for tok, pos in pos_tags]

print('Filtered Tokens:\n', sample_filtered)
print('\nLemmatized Tokens:\n', lemmatize_tokens(sample_filtered))

Filtered Tokens:
 ['amazing', 'stay', 'place', 'felt', 'cozy', 'guests', 'checkin', 'smooth', 'amenities', 'exactly', 'needed']

Lemmatized Tokens:
 ['amaze', 'stay', 'place', 'felt', 'cozy', 'guest', 'checkin', 'smooth', 'amenity', 'exactly', 'need']


In [9]:
comparison = pd.DataFrame({
    'original': sample_filtered,
    'stemmed': stem_tokens(sample_filtered),
    'lemmatized': lemmatize_tokens(sample_filtered)
})
comparison

,original,stemmed,lemmatized
0,amazing,amaz,amaze
1,stay,stay,stay
2,place,place,place
3,felt,felt,felt
4,cozy,cozi,cozy
5,guests,guest,guest
6,checkin,checkin,checkin
7,smooth,smooth,smooth
8,amenities,amen,amenity
9,exactly,exactli,exactly


In [10]:
def preprocess_review_v2(text):
    """Pipeline: tokenize -> filter (incl. stopword removal) -> stem -> lemmatize."""
    _, word_tokens = tokenize_text(text)
    filtered = filter_tokens(word_tokens)
    stemmed = stem_tokens(filtered)
    lemmatized = lemmatize_tokens(filtered)
    return pd.Series({
        'filtered_tokens': filtered,
        'stemmed_tokens': stemmed,
        'lemmatized_tokens': lemmatized
    })

processed_v2 = df['review_text'].apply(preprocess_review_v2)
df_processed = pd.concat([df[['review_id', 'review_text']], processed_v2], axis=1)
df_processed.head(10)

,review_id,review_text,filtered_tokens,stemmed_tokens,lemmatized_tokens
0,369314882,Amazing stay! The place felt very cozy for 4 g...,"[amazing, stay, place, felt, cozy, guests, che...","[amaz, stay, place, felt, cozi, guest, checkin...","[amaze, stay, place, felt, cozy, guest, checki..."
1,490116563,It was okay for the price. Location in XIII Au...,"[okay, price, location, xiii, aurelia, conveni...","[okay, price, locat, xiii, aurelia, conveni, e...","[okay, price, location, xiii, aurelia, conveni..."
2,582235668,Loved every minute of it. Our superhost was su...,"[loved, every, minute, superhost, super, respo...","[love, everi, minut, superhost, super, respons...","[love, every, minute, superhost, super, respon..."
3,68054683,Decent stay overall. Some things could be impr...,"[decent, stay, overall, things, could, improve...","[decent, stay, overal, thing, could, improv, l...","[decent, stay, overall, thing, could, improve,..."
4,248483824,Reasonable for a short trip. Location in Long ...,"[reasonable, short, trip, location, long, isla...","[reason, short, trip, locat, long, island, cit...","[reasonable, short, trip, location, long, isla..."
5,155617131,Decent stay overall. It served its purpose for...,"[decent, stay, overall, served, purpose, stay,...","[decent, stay, overal, serv, purpos, stay, pari]","[decent, stay, overall, serve, purpose, stay, ..."
6,710244614,"Nothing special, but fine. Our superhost was p...","[nothing, special, fine, superhost, polite, sl...","[noth, special, fine, superhost, polit, slow, ...","[nothing, special, fine, superhost, polite, sl..."
7,299174484,We had a rough experience. The location in Enc...,"[rough, experience, location, enclosstlaurent,...","[rough, experi, locat, enclosstlaur, noisier, ...","[rough, experience, location, enclosstlaurent,..."
8,22136604,Perfect for our trip. Check-in was smooth and ...,"[perfect, trip, checkin, smooth, amenities, ex...","[perfect, trip, checkin, smooth, amen, exactli...","[perfect, trip, checkin, smooth, amenity, exac..."
9,469473761,Would not recommend. The private room in house...,"[would, recommend, private, room, house, clean...","[would, recommend, privat, room, hous, clean, ...","[would, recommend, private, room, house, clean..."


In [11]:
df_processed.to_csv('Preprocessed_Airbnb_Reviews_v2.csv', index=False)
print('Saved to Preprocessed_Airbnb_Reviews_v2.csv')

Saved to Preprocessed_Airbnb_Reviews_v2.csv
